# 🛡️ Custom Ridge Regression (Closed-Form & Gradient Descent)

A custom implementation of **L2-regularized Linear Regression (Ridge)** built from scratch in NumPy, supporting both the **Closed-Form Normal Equation** and **Gradient Descent** optimization, with a Scikit-Learn–compatible API (`fit`, `predict`, `coef_`, `intercept_`).

---

## 🎯 Project Goal
Build a flexible `CustomRidgeRegressor` class from first principles that prevents overfitting and handles multicollinearity by penalizing large feature weights:
- **`alpha`**: Regularization strength ($\lambda \ge 0$) controlling L2 penalty intensity (default: `1.0`)
- **`solver`**: `'closed_form'` (Normal Equation) or `'gd'` (Batch Gradient Descent) (default: `'closed_form'`)
- **`learning_rate`**: Step size for gradient updates when `solver='gd'` (default: `0.01`)
- **`epoch`**: Number of iterations over the training set when `solver='gd'` (default: `1000`)

---

## 🧮 Mathematical Formulation

Ridge Regression adds an **L2 penalty** (proportional to the squared magnitude of the feature weights) to the Mean Squared Error loss function, while **excluding the intercept ($w_0$)** from regularization:

$$J(w, b) = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}_i - y_i)^2 + \frac{\lambda}{m} \sum_{j=1}^{n} w_j^2$$

| Solver Method | Mathematical Update / Solution | Key Implementation Detail |
| :--- | :--- | :--- |
| **Closed-Form (`closed_form`)** | $w = (X^T X + \lambda I)^{-1} X^T y$ | Prepend bias column of `1`s to $X$ and set `I[0, 0] = 0` so intercept $w_0$ is unpenalized|
| **Gradient Descent (`gd`)** | $w := w - \eta \left( -\frac{2}{m} X^T (y - \hat{y}) + \frac{2\lambda}{m} w \right)$ | Intercept gradient $-\frac{2}{m}\sum(y - \hat{y})$ remains unchanged (no L2 term on $b$)

---

## ✅ Pass Conditions & Verification
1. **Scikit-Learn Equivalence:** Verify that both `solver='closed_form'` and `solver='gd'` match `sklearn.linear_model.Ridge(alpha=...)` coefficients and intercept on standardized synthetic data (`StandardScaler` + `make_regression`)
2. **Regularization Shrinkage Sweep ($\lambda \in [0, 1, 10, 100]$):** Sweep `alpha` across `[0, 1, 10, 100]` and confirm that
   - At $\lambda = 0$, weights match Ordinary Least Squares (`LinearRegression`)
   - As $\lambda$ increases, all feature weights shrink smoothly toward zero **without becoming exact zeros** (unlike L1/Lasso)

---

## 🚀 Example Usage

```python
import numpy as np
from sklearn.datasets import make_regression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# 1. Generate and standardize synthetic data (scaling is mandatory for Ridge)
X, y = make_regression(n_samples=200, n_features=5, noise=15, random_state=42)
X_scaled = StandardScaler().fit_transform(X)

# 2. Fit custom Ridge models (Closed-Form & Gradient Descent)
ridge_cf = CustomRidgeRegressor(alpha=10.0, solver='closed_form').fit(X_scaled, y)
ridge_gd = CustomRidgeRegressor(alpha=10.0, solver='gd', learning_rate=0.01, epoch=1000).fit(X_scaled, y)

# 3. Verify against Scikit-Learn Ridge
sk_ridge = Ridge(alpha=10.0).fit(X_scaled, y)

print("Closed-Form Coef:", np.round(ridge_cf.coef_, 4))
print("Grad Descent Coef:", np.round(ridge_gd.coef_, 4))
print("Sklearn Ridge Coef:", np.round(sk_ridge.coef_, 4))

In [8]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

In [9]:
class CustomRidgeRegressor():
    '''alpha --> Select the alpha or lambda parameter 
    
        solver -- > 'gd' for Gradient Descent approach 
                    'closed_form' for Ordinary Least Square approach (default)
                    
        epoch --> if solver = 'gd' : set the number of iteration 
        learning_rate --> if solver = 'gd' : set the learning rate
    '''

    def __init__(self,alpha=0.1,solver='closed_form',epoch=1000,learning_rate=0.01):

        self.alpha = alpha
        self.solver = solver
        self.epoch = epoch
        self.lr = learning_rate
        self.Beta = None
        self.coef_ = None
        self.intercept_ = None
        
        val_solver = ['closed_form','gd']
        if self.solver not in val_solver:
            raise ValueError(f'{self.solver} is not a valid solver. Expected {val_solver}' )

    def fit(self,X,Y):
        
        # converting dataframes into arrays
        X = np.asarray(X)
        Y = np.asarray(Y) 

        # adding a bias column in our input array
        X = np.insert(X,0,1,axis=1)
        
        if self.solver == 'closed_form':    # Ordinary Least Square Approach

            I = np.identity(X.shape[1])   # creating an identity matrix 
            I[0][0] = 0  # initialising bias term to zero

            a = np.linalg.inv(np.dot(X.T,X) + self.alpha*I)
            b = np.dot(X.T,Y)

            self.Beta = np.dot(a,b)
            self.coef_ = self.Beta[1:]
            self.intercept_ = self.Beta[0]

        if self.solver == 'gd':

            self.Beta = np.zeros(X.shape[1]) # initial values

            for i in range(self.epoch):

                # not penalizing the intercept
                reg = self.alpha*self.Beta
                reg[0] = 0
                
                # Divide by m (or multiply by 2/m) so lr=0.01 is stable
                beta_der = (-X.T@Y + np.dot((X.T@X),self.Beta) + reg)/X.shape[0]
                
                self.Beta = self.Beta - self.lr*beta_der
                self.coef_ = self.Beta[1:]
                self.intercept_ = self.Beta[0]
                
        return self

    def predict(self,x_test):
        if self.Beta is None:
            raise ValueError('Cannot predict call fit first')
        # adding a bias column in our test array
        x_test = np.insert(x_test,0,1,axis=1)
        return np.dot(x_test,self.Beta)

In [10]:
X, Y = make_regression(n_samples=200, n_features=5, noise=15, random_state=42)

from sklearn.model_selection import train_test_split
X_train,x_test,Y_train,y_test = train_test_split(X,Y,test_size=0.2,random_state=2)

In [11]:
ridge = Ridge(alpha = 0.1)
ridge.fit(X_train,Y_train)
print('coef: ',ridge.coef_)
print('intercept: ',ridge.intercept_)

coef:  [ 3.3822737  10.74004433 64.12328605 17.88114827 68.91870137]
intercept:  1.2101515213508791


In [12]:
my_ridge = CustomRidgeRegressor()
my_ridge.fit(X_train,Y_train)
print('coef: ',my_ridge.coef_)
print('intercept: ',my_ridge.intercept_)

coef:  [ 3.3822737  10.74004433 64.12328605 17.88114827 68.91870137]
intercept:  1.2101515213508796


In [13]:
my_ridge_gd = CustomRidgeRegressor(solver='gd')
my_ridge_gd.fit(X_train,Y_train)
print('coef: ',my_ridge_gd.coef_)
print('intercept: ',my_ridge_gd.intercept_)

coef:  [ 3.38395304 10.73873872 64.11750391 17.88168283 68.91464168]
intercept:  1.2098653014282221


In [14]:
print('R2 Score of ridge: ',r2_score(y_test,ridge.predict(x_test)))
print('R2 Score of my_ridge: ',r2_score(y_test,my_ridge.predict(x_test)))
print('R2 Score of my_ridge_gd: ',r2_score(y_test,my_ridge_gd.predict(x_test)))

R2 Score of ridge:  0.9743281020322349
R2 Score of my_ridge:  0.9743281020322347
R2 Score of my_ridge_gd:  0.974322704969239


In [15]:
# 1. Generate and standardize synthetic data (scaling is mandatory for Ridge)
X, y = make_regression(n_samples=200, n_features=5, noise=15, random_state=42)
X_scaled = StandardScaler().fit_transform(X)

# 2. Fit custom Ridge models (Closed-Form & Gradient Descent)
ridge_cf = CustomRidgeRegressor(alpha=10.0, solver='closed_form').fit(X_scaled, y)
ridge_gd = CustomRidgeRegressor(alpha=10.0, solver='gd', learning_rate=0.01, epoch=1000).fit(X_scaled, y)

# 3. Verify against Scikit-Learn Ridge
sk_ridge = Ridge(alpha=10.0).fit(X_scaled, y)

print("Closed-Form Coef:", np.round(ridge_cf.coef_, 4))
print("Grad Descent Coef:", np.round(ridge_gd.coef_, 4))
print("Sklearn Ridge Coef:", np.round(sk_ridge.coef_, 4))

Closed-Form Coef: [ 3.5799 10.2919 59.1925 16.2017 64.5789]
Grad Descent Coef: [ 3.5812 10.2919 59.1908 16.2017 64.5778]
Sklearn Ridge Coef: [ 3.5799 10.2919 59.1925 16.2017 64.5789]


In [16]:
# Sweep lambda across [0, 1, 10, 100]
for lam in [0, 1, 10, 100]:
    model = CustomRidgeRegressor(alpha=lam, solver='closed_form').fit(X_scaled, y)
    print(f"alpha={lam:<3} -> coef_: {np.round(model.coef_, 4)}")

alpha=0   -> coef_: [ 3.5041 10.7344 62.0172 16.9718 67.7525]
alpha=1   -> coef_: [ 3.5131 10.6884 61.7225 16.8916 67.4212]
alpha=10  -> coef_: [ 3.5799 10.2919 59.1925 16.2017 64.5789]
alpha=100 -> coef_: [ 3.5432  7.511  42.0284 11.4902 45.4158]
